In [ ]:
PATH_Virus= "C://Users//chien//Downloads//CyAI_Malware//AiDetect//Release//DataAI//DataVirusNet"
PATH_Clean = "C://Users//chien//Downloads//CyAI_Malware//AiDetect//Release//DataAI//DataCleansNet"

In [ ]:
import os
from sklearn.model_selection import train_test_split

clean_files = [os.path.join(PATH_Clean, f) for f in os.listdir(PATH_Clean)]
virus_files = [os.path.join(PATH_Virus, f) for f in os.listdir(PATH_Virus)]

y_clean = [0] * len(clean_files)  # Nhãn 0 cho các tệp clean
y_virus = [1] * len(virus_files)  # Nhãn 1 cho các tệp virus


# Kết hợp Clean và Virus vào tập huấn luyện và kiểm tra
X_train = clean_files + virus_files
y_train = y_clean + y_virus


# In ra thông tin về số lượng tệp trong tập train và test
print(f"Số lượng file trong tập train: {len(X_train)}")
print(f"Số lượng file trong tập test: {len(y_train)}")


In [ ]:
from sklearn.model_selection import train_test_split

# Chia tập dữ liệu thành 80% train và 20% test
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.5, random_state=42)

In [ ]:
len(X_train)

In [ ]:
len(X_train)

In [ ]:
X_train=X_train[:80000]
X_test=X_test[:80000]
y_train=y_train[:80000]
y_test=y_test[:80000]


In [ ]:
def splitData(filepath):
    with open(filepath, 'r',encoding ='latin-1') as file:
        # Đọc tất cả dữ liệu trong file
        data_compact = file.read().strip()
        data_split = data_compact.split('\t')  
    return data_split

In [ ]:
# Lấy nhưng tệp dữ liệu chứa độ dài khác với độ dài tiêu chuẩn cho hết vào listpop
dem =0
count = 0
listpop=[]
for path in X_train:
    data_split = splitData(path)
    if len(data_split)!=796 and len(data_split)>540 :
        print(f"{path}:{count}:{len(data_split)}")
        listpop.append(count)
        dem=dem+1
    count = count + 1
print(dem)

In [ ]:
# Lấy nhưng tệp dữ liệu chứa độ dài khác với độ dài tiêu chuẩn cho hết vào listpopTest
dem =0
count = 0
listpopTest =[]
for path in X_test:
    data_split = splitData(path)
    if len(data_split)!=796 and len(data_split)>540 :
        print(f"{path}:{count}:{len(data_split)}")
        listpopTest.append(count)
        dem=dem+1
    count = count + 1
print(dem)

In [ ]:
len(X_train)

In [ ]:
# Lưu ý cần chạy code này để remove các đưong dan chứa file có nhiều hơn 796 đặc trưng
for fe in sorted(listpop, reverse=True):
    X_train.pop(fe)
    y_train.pop(fe)

In [ ]:
len(y_test)

In [ ]:
# Lưu ý cần chạy code này để remove các đưong dan chứa file có nhiều hơn 796 đặc trưng
for fe in sorted(listpopTest, reverse=True):
    X_test.pop(fe)
    y_test.pop(fe)

In [ ]:
len(y_test)

In [ ]:
data_split[:795]

# Xử lý section


In [ ]:
import re
def sections(data_split):
  if len(data_split) == 540:
      new_elements = ['0.0'] * 256
      data_split = data_split[:292] + new_elements + data_split[292:]
  data_sections = data_split[794].lower()
  sections = data_sections.split()
  newSection =[]
  for i in range(10):
    if len(sections) < 10:
        if i>=len(sections):
            newSection.append('null')
        else:
            newSection.append(sections[i]) 
    else:
            newSection.append(sections[i])
  return newSection

In [ ]:
# trả về dữ liệu chứa các section,đã được padding
def data_section(listPath):
    list_section =[]
    for path in listPath:
        data_split = splitData(path) # cắt theo dấu \t
        section = sections(data_split) 
        list_section.append(section)
    return list_section

In [ ]:
# xây từ điển dict1 cho section
def buildVocabSection(X_train):
    list_section =data_section(X_train)
    setSection=[]
    count= 0
    dict1={}
    for sublis in list_section:
        for item in sublis:
            if item !="null" and item not in setSection:
                setSection.append(item)
                dict1 [item]=count
            if item in setSection:
                dict1[item] = dict1[item] + 1
    dict1['null']=0
    sorted_dict = dict(sorted(dict1.items(), key=lambda x: x[1], reverse=True))
    print(sorted_dict)
    vector_format={}
    count = 1
    for key,value in sorted_dict.items():
        vector_format[key]=count
        count = count + 1
    return vector_format

In [ ]:
vector_format = buildVocabSection(X_train) 

In [ ]:
vector_format['null']=0
print(vector_format)

In [ ]:
with open("PENETSection", "w") as file:
    # Ghi các key ngăn cách nhau bởi dấu cách
    file.write(" ".join(vector_format.keys()))

In [ ]:
list_section_train = data_section(X_train)
list_section_test = data_section(X_test)

In [ ]:
def getVectorSection(list_section,vector_format):
    vector_section = []
    for line in list_section:
        vector_line = []
        print(line)
        for i in range(10):
            try:
              vector_line.append(vector_format[line[i]])
            except:
              vector_line.append(list(vector_format.values())[-1]+1)
              print(line[i])
              vector_format[line[i]]= list(vector_format.values())[-1]+1
              pass
        vector_section.append(vector_line)
    return vector_section
    

In [ ]:
vector_section_train = getVectorSection(list_section_train,vector_format)
vector_section_test = getVectorSection(list_section_test,vector_format)

In [ ]:
len(vector_section_train)

In [ ]:
def data_IP(data_split):
  if len(data_split) == 540:
      new_elements = ['0.0'] * 256
      data_split = data_split[:292] + new_elements + data_split[292:]
  data_IP = data_split[795:]
  combined_string = ''.join(data_IP)
  data_import = re.sub(r'[\n\t]', '', combined_string).lower()
  data_import = data_import.split()
  return data_import

In [ ]:
# trả về list dữ liệu chứa các hàm import
def data_imports(listPath):
    list_ip =[]
    for path in listPath:
        data_split = splitData(path) # cắt theo dấu \t
        data_ip = data_IP(data_split) 
        list_ip.append(data_ip)
    return list_ip

In [ ]:
list_ip_train = data_imports(X_train)
list_ip_test =data_imports(X_test)

In [ ]:
len(list_ip_train[1])

In [ ]:
len(y_train)

In [ ]:
from collections import Counter
# hàm compute_scores_and_filter đầu vào là 1 list chứa các list con chứa các tên hàm import và 1 list là nhãn tương ứng đối với từng file 
# Mục đích trả về danh sách các từ thỏa mãn với điều kiện dưới, để lấy ra được các hàm hay gặp nhưng không quá là phổ biến ở cả tệp cleans và virus
def compute_scores_and_filter(list_ip_train, labels):
    counts_virus = Counter()
    counts_clean = Counter()
    
    # Đếm số lần xuất hiện của từ trong từng nhóm
    for doc, label in zip(list_ip_train, labels):
        if label == 1:  # Virus
            counts_virus.update(doc)
        else:  # Sạch
            counts_clean.update(doc)
    
    # Tìm từ thỏa mãn điều kiện
    filtered_terms = []
    for term in set(counts_virus.keys()).union(set(counts_clean.keys())):
        # Điều kiện: từ có độ dài lớn hơn 6 ký tự
        if len(term) > 6:
            count_virus = counts_virus[term]
            count_clean = counts_clean[term]
            
            # Điều kiện: xuất hiện trong file virus nhiều hơn gấp đôi file sạch
            # Và hiệu giữa số lần xuất hiện trong virus và sạch >= 40
            if (count_virus - count_clean) >= 5000 and count_clean >=0 and count_clean < 50 :
            # if count_virus  >= 1 and count_virus <=3  and count_clean ==0:
                filtered_terms.append(term)
            elif count_clean>50 and count_virus>10*count_clean:
                filtered_terms.append(term)

    return filtered_terms

# Gọi hàm để lấy danh sách từ thỏa mãn điều kiện
filtered_terms = compute_scores_and_filter(list_ip_train, y_train)

# Ghi từ điển vào file
with open("PENETDICT", "a") as f:
    for term in filtered_terms:
        f.write(term + " ") 

In [ ]:
# Khởi tạo từ điển vocab
vocab = []
    
# Đọc file và xử lý  
with open("PENETDICT", "r") as file:
    content = file.read()  # Đọc toàn bộ nội dung file 
    vocab = content.split()  # Cắt nội dung theo khoảng trắng

In [ ]:
len(vocab)

In [ ]:
list_ip_virus = []
for ip_train,label in zip(list_ip_train,y_train):
    if label ==1:
        list_ip_virus.append(ip_train)

In [ ]:
from collections import Counter

# Danh sách chứa các list con với các hàm Windows API


# Gộp tất cả các list con lại thành một list duy nhất
all_api_calls = [api for sublist in list_ip_virus for api in sublist if api != 'getstringnull']

# Bước 2: Đếm tần suất xuất hiện của từng hàm
api_counter = Counter(all_api_calls)

# Bước 3: Sắp xếp các hàm API theo tần suất giảm dần và gán giá trị
sorted_apis = sorted(api_counter, key=api_counter.get, reverse=True)

# Xây dựng từ điển với giá trị là thứ tự xuất hiện
api_ranking = {api: rank + 1 for rank, api in enumerate(sorted_apis)}


In [ ]:
api_ranking

In [ ]:
key = list(api_ranking.keys())[0]
vocab = []
for i in range(500,1500):
    key = list(api_ranking.keys())[i]
    vocab.append(key)

In [ ]:
vocab

In [ ]:
from collections import Counter

def compute_scores_and_filter(list_ip_train, labels):
    counts_virus = Counter()
    counts_clean = Counter()
    
    # Đếm số lần xuất hiện của từ trong từng nhóm
    for doc, label in zip(list_ip_train, labels):
        if label == 1:  # Virus
            counts_virus.update(doc)
        else:  # Sạch
            counts_clean.update(doc)
    
    # Tìm từ thỏa mãn điều kiện
    filtered_terms = []
    for term in set(counts_virus.keys()).union(set(counts_clean.keys())):
        # Điều kiện: từ có độ dài lớn hơn 6 ký tự
        if len(term) > 6:
            count_virus = counts_virus[term]
            count_clean = counts_clean[term]
            
            # Điều kiện: xuất hiện trong file virus nhiều hơn gấp đôi file sạch
            # Và hiệu giữa số lần xuất hiện trong virus và sạch >= 40
            if count_clean >= 3 * count_virus and (count_clean - count_virus) >= 3000 and count_virus > 1 and count_virus < 5000:
                filtered_terms.append(term)
    
    return filtered_terms

# Gọi hàm để lấy danh sách từ thỏa mãn điều kiện
filtered_terms = compute_scores_and_filter(list_ip_train, y_train)

# Ghi từ điển vào file
with open("DictNet", "a") as f:
    for term in filtered_terms:
        f.write(term + " ")


In [ ]:
len(vocab)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
def getVectorIP(list_ip):
  vector_import = []
  for line in list_ip:
      vector_line1 = []
      for i in vocab:
            if i in line:
              vector_line1.append(1)
            else:
              vector_line1.append(0)
      vector_import.append(vector_line1)
  return vector_import


In [ ]:
vector_import_train = getVectorIP(list_ip_train)
vector_import_test = getVectorIP(list_ip_test)

In [ ]:
len(vector_import_train[1])

In [ ]:
vector_import_test.tolist()

In [ ]:
vector_import_train.tolist()

In [ ]:
# Lấy dữ liệu số 
def getDataFloat(X):
    datafloat = []
    for file in X:
        data_origin = splitData(file)
        if len(data_origin) == 540:
            new_elements = ['0.0'] * 256
            data_origin = data_origin[:292] + new_elements + data_origin[292:]
        if len(data_origin) < 796:
            break
        datafloat.append(data_origin[:794])
    return datafloat

In [ ]:
# chuyển hết sang float
def convertFloat(datafloat,vector_section,vector_import):
    for i in range(len(datafloat)):
        datafloat[i] = list(map(float, datafloat[i]))
        vector_section[i] = list(map(float, vector_section[i]))
        vector_import[i] = list(map(float, vector_import[i]))
    return datafloat,vector_section,vector_import

In [ ]:
# Gộp 3 vector chuẩn bị đưa vào training
def mergeData(datafloat,vector_section,vector_import):
    for i in range(len(datafloat)):
        datafloat[i].extend(vector_section[i])
        datafloat[i].extend(vector_import[i])
    return datafloat

In [ ]:
datafloat_train=getDataFloat(X_train)
datafloat_test=getDataFloat(X_test)


In [ ]:
datafloat_train,vector_section_train,vector_import_train = convertFloat(datafloat_train,vector_section_train,vector_import_train)
datafloat_test,vector_section_test,vector_import_test = convertFloat(datafloat_test,vector_section_test,vector_import_test)

In [ ]:
data_train =mergeData(datafloat_train,vector_section_train,vector_import_train)
data_test = mergeData(datafloat_test,vector_section_test,vector_import_test)

In [ ]:
len(data_train[1])

In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# # Lưu mô hình vào file model.mdl
with open('CYAIPENET.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
# model = clf.booster_

# # # Lưu mô hình vào file model.mdl
# # with open('CYAIPENET.mdl', 'wb') as file:
# #     pickle.dump(model, file)


In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
# model = clf.booster_

# # # Lưu mô hình vào file model.mdl
# # with open('CYAIPENET.mdl', 'wb') as file:
# #     pickle.dump(model, file)


In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# Lưu mô hình vào file model.mdl
with open('CYAIPENET.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# Lưu mô hình vào file model.mdl
with open('CYAIPENET.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
import numpy as np

# Lấy mức độ quan trọng của các đặc trưng từ mô hình đã huấn luyện
feature_importances = clf.feature_importances_
total_importance = np.sum(feature_importances)
dactrung4 = []
# Tính tỷ lệ mức độ quan trọng (phần trăm)
feature_importance_ratios = (feature_importances / total_importance) * 100

# Xử lý tên đặc trưng nếu data_train là list
if isinstance(data_train, list):
    num_features = len(data_train[0])  # Số cột dựa trên hàng đầu tiên
else:
    num_features = data_train.shape[1]  # Nếu là numpy array hoặc DataFrame

feature_names = [i for i in range(num_features)]
a= 0
for feature, importance_ratio in zip(feature_names, feature_importance_ratios):
    if round(importance_ratio, 4) > 0.003 and a>785:
        dactrung4.append(feature)
    a = a+1
# In từng phần trăm của từng đặc trưng
for feature, importance_ratio in zip(feature_names, feature_importance_ratios):
    print(f"{feature}: {importance_ratio:.5f}%")
